In [1]:
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 xformers --index-url https://download.pytorch.org/whl/cu124

!git clone https://github.com/resemble-ai/chatterbox.git
%cd chatterbox
!pip install -e .
%cd ..

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.1 MB/s eta 0:00:0031m95.5 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 101.3 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 120.9 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 127.5 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 125.5 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 120.9 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 117.9 MB/s eta 0:00:00m eta 0:00:010:00:01
  Using cached https://download.pytorch.org/whl/nvidia_nccl_cu12-2.2

In [2]:
%cd chatterbox

/home/anivan41/lep-speech/chatterbox


In [3]:
import torchaudio as ta
from src.chatterbox.tts import ChatterboxTTS

model = ChatterboxTTS.from_pretrained(device="cuda")


ERROR! Session/line number was not unique in database. History logging moved to new session 32


ImportError: /opt/conda/lib/python3.11/site-packages/torch/lib/../../cusparselt/lib/libcusparseLt.so.0: cannot read file data

In [3]:
import torchaudio as ta
from src.chatterbox.tts import ChatterboxTTS

model = ChatterboxTTS.from_pretrained(device="cuda")


ERROR! Session/line number was not unique in database. History logging moved to new session 32


ImportError: /opt/conda/lib/python3.11/site-packages/torch/lib/../../cusparselt/lib/libcusparseLt.so.0: cannot read file data

In [ ]:
import shutil
from pathlib import Path

def copy_dereference(src_dir: str, dst_dir: str):
    src = Path(src_dir)
    dst = Path(dst_dir)

    # Ensure destination is a clean directory
    if dst.exists() and dst.is_file():
        dst.unlink()  # remove if it's accidentally a file
    dst.mkdir(parents=True, exist_ok=True)

    for item in src.iterdir():
        if item.name.startswith('.'):
            continue

        if item.is_symlink() or item.is_file():
            try:
                real_src = item.resolve()
                if real_src.is_file():
                    # Step 1: Copy into dst (uses blob hash as filename)
                    copied_path = shutil.copy2(real_src, dst)
                    # Step 2: Rename to the symlink's original name
                    new_path = dst / item.name
                    # If file already exists, overwrite it
                    if new_path.exists():
                        new_path.unlink()
                    Path(copied_path).rename(new_path)
                    print(f"✅ Copied as: {item.name}")
                else:
                    print(f"⚠️ Skipping non-file: {item.name}")
            except Exception as e:
                print(f"❌ Failed on {item.name}: {e}")
        else:
            print(f"📁 Skipping directory: {item.name}")

# Usage
copy_dereference(
    "/home/aali2/.cache/huggingface/hub/models--ResembleAI--chatterbox/snapshots/05e904af2b5c7f8e482687a9d7336c5c824467d9",
    "/home/aali2/TTS_low_resource/chatterbox/temp_model"
)

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers
from datasets import load_dataset
import os
import tempfile

# -------------------------------
# 1. Load the dataset
# -------------------------------

print("📥 Loading the dataset")

try:
    dataset = load_dataset("Vikhrmodels/ToneSpeak", split="train")
except Exception as e:
    raise RuntimeError(f"Failed to load 'ToneSpeak' dataset. Make sure you're logged in to Hugging Face. Error: {e}")

print(f"✅ Loaded {len(dataset)} examples.")

# Extract text 

def extract_text(example, name="text"):
    return {"text": example[name]}

# Apply mapping to extract text
print("🧹 Extracting text with tashkeel...")
texts = dataset.map(extract_text, num_proc=4, remove_columns=dataset.column_names)


# Save to temporary file for tokenizer training
temp_dir = tempfile.mkdtemp()
temp_file = os.path.join(temp_dir, "ru.txt")

print("🧹 Saving Russian text to file...")

with open(temp_file, "w", encoding="utf-8") as f:
    for i in range(len(texts)):
        example = texts[i]  # This ensures we get a dict
        text = example["text"].strip()
        if text:
            f.write(text + "\n")
        if i % 5000 == 0:
            print(f"📝 Written {i} lines")

print(f"✅ Text saved to {temp_file}")

# -------------------------------
# 2. Initialize BPE Tokenizer
# -------------------------------

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

# -------------------------------
# 3. Configure BPE Trainer (vocab_size = 2000)
# -------------------------------

trainer = trainers.BpeTrainer(
    vocab_size=300,
    min_frequency=1000,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    show_progress=True,
)

# -------------------------------
# 4. Train the Tokenizer
# -------------------------------

print("🚀 Training BPE tokenizer on Russian text ")

tokenizer.train(files=[temp_file], trainer=trainer)

# -------------------------------
# 5. Save the Tokenizer
# -------------------------------

output_dir = "./ru_bpe_2000"
os.makedirs(output_dir, exist_ok=True)
tokenizer.save(f"{output_dir}/tokenizer.json")
tokenizer.model.save(output_dir)  # Saves vocab.json and merges.txt

print(f"✅ Tokenizer trained and saved to '{output_dir}'")
print(f"📊 Final vocabulary size: {tokenizer.get_vocab_size()}")

# -------------------------------
# 7. Cleanup
# -------------------------------

import shutil
shutil.rmtree(temp_dir)
print(f"🧹 Temporary data directory removed.")

In [ ]:
from tokenizers import Tokenizer

# Load the saved tokenizer
tokenizer_path = "./ru_bpe_2000/tokenizer.json"
tokenizer = Tokenizer.from_file(tokenizer_path)

print("✅ Tokenizer loaded successfully!")
print(f"📊 Vocabulary size: {tokenizer.get_vocab_size()}")
ids = tokenizer.encode("привет").ids
tokenizer.decode(ids), ids

In [ ]:
from src.chatterbox.models.tokenizers.tokenizer import EnTokenizer
tokenizer_orig = EnTokenizer("./temp_model/tokenizer.json")
import json
with open("./temp_model/tokenizer.json", "r") as f:
    state = json.load(f)
state2 = json.loads(tokenizer.__getstate__())
state2.keys()  


vocab = state["model"]["vocab"]  # {token: index}
vocab2 = state2["model"]["vocab"]  # {token: index}
for key in vocab2.keys():
    vocab2[key] += 224
merges1 = state["model"]["merges"]  # {token: index}
merges2 = state2["model"]["merges"]  # {token: index}
def merge_bpe_merges(merges1, merges2):
    """
    Merge two BPE merge lists:
    - merges1: list of strings like "a b"
    - merges2: list of lists like ['a', 'b']
    Returns: list of strings in format "a b", no duplicates, order preserved.
    """
    seen = set()
    merged = []

    # Add merges1
    for merge in merges1:
        if merge not in seen:
            seen.add(merge)
            merged.append(merge)

    # Add merges2 (convert ['a', 'b'] -> "a b")
    for pair in merges2:
        merge_str = " ".join(pair)
        if merge_str not in seen:
            seen.add(merge_str)
            merged.append(merge_str)

    return merged

# Rebuild vocab with new indices
new_vocab = {}
next_id = 0
# Add all tokens from vocab1 first
for token in vocab:
    if token not in new_vocab:
        new_vocab[token] = next_id
        next_id += 1
# Add tokens from vocab2, skipping duplicates
for token in vocab2:
    if token not in new_vocab:
        new_vocab[token] = next_id
        next_id += 1
state["model"]["vocab"] = new_vocab

state["model"]["merges"] = merge_bpe_merges(merges1, merges2)

with open("en_ru_tokenizer.json", "w") as f:
    json.dump(state,f)

In [ ]:
tokenizer_new = EnTokenizer("./en_ru_tokenizer.json")

In [ ]:
def transliterate_ru_to_latin(text: str) -> str:
    """
    Transliterates Russian Cyrillic text to Latin using GOST 7.79-2000 (System B).
    
    Args:
        text (str): Input string in Russian (Cyrillic).
    
    Returns:
        str: Transliterated string in Latin characters.
    """
    mapping = {
        'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd',
        'е': 'e', 'ё': 'yo', 'ж': 'zh', 'з': 'z', 'и': 'i',
        'й': 'j', 'к': 'k', 'л': 'l', 'м': 'm', 'н': 'n',
        'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't',
        'у': 'u', 'ф': 'f', 'х': 'kh', 'ц': 'ts', 'ч': 'ch',
        'ш': 'sh', 'щ': 'shh', 'ъ': '', 'ы': 'y', 'ь': '',
        'э': 'e', 'ю': 'yu', 'я': 'ya',
        'А': 'A', 'Б': 'B', 'В': 'V', 'Г': 'G', 'Д': 'D',
        'Е': 'E', 'Ё': 'Yo', 'Ж': 'Zh', 'З': 'Z', 'И': 'I',
        'Й': 'J', 'К': 'K', 'Л': 'L', 'М': 'M', 'Н': 'N',
        'О': 'O', 'П': 'P', 'Р': 'R', 'С': 'S', 'Т': 'T',
        'У': 'U', 'Ф': 'F', 'Х': 'Kh', 'Ц': 'Ts', 'Ч': 'Ch',
        'Ш': 'Sh', 'Щ': 'Shh', 'Ъ': '', 'Ы': 'Y', 'Ь': '',
        'Э': 'E', 'Ю': 'Yu', 'Я': 'Ya'
    }

    return ''.join(mapping.get(char, char) for char in text)

In [ ]:
def transliterate_ru_to_latin(text: str) -> str:
    """
    Transliterates Russian Cyrillic text to Latin using GOST 7.79-2000 (System B).
    
    Args:
        text (str): Input string in Russian (Cyrillic).
    
    Returns:
        str: Transliterated string in Latin characters.
    """
    mapping = {
        'а': 'a', 'б': 'b', 'в': 'v', 'г': 'g', 'д': 'd',
        'е': 'e', 'ё': 'yo', 'ж': 'zh', 'з': 'z', 'и': 'i',
        'й': 'j', 'к': 'k', 'л': 'l', 'м': 'm', 'н': 'n',
        'о': 'o', 'п': 'p', 'р': 'r', 'с': 's', 'т': 't',
        'у': 'u', 'ф': 'f', 'х': 'kh', 'ц': 'ts', 'ч': 'ch',
        'ш': 'sh', 'щ': 'shh', 'ъ': '', 'ы': 'y', 'ь': '',
        'э': 'e', 'ю': 'yu', 'я': 'ya',
        'А': 'A', 'Б': 'B', 'В': 'V', 'Г': 'G', 'Д': 'D',
        'Е': 'E', 'Ё': 'Yo', 'Ж': 'Zh', 'З': 'Z', 'И': 'I',
        'Й': 'J', 'К': 'K', 'Л': 'L', 'М': 'M', 'Н': 'N',
        'О': 'O', 'П': 'P', 'Р': 'R', 'С': 'S', 'Т': 'T',
        'У': 'U', 'Ф': 'F', 'Х': 'Kh', 'Ц': 'Ts', 'Ч': 'Ch',
        'Ш': 'Sh', 'Щ': 'Shh', 'Ъ': '', 'Ы': 'Y', 'Ь': '',
        'Э': 'E', 'Ю': 'Yu', 'Я': 'Ya'
    }

    return ''.join(mapping.get(char, char) for char in text)

In [ ]:
from safetensors import safe_open
from safetensors.torch import save_file
def load_safetensors_to_torch(file_path: str, device: str = "cpu") -> dict:
    """Load a safetensors file into a PyTorch state dict."""
    print(f"Loading safetensors file: {file_path}")
    state_dict = {}
    with safe_open(file_path, framework="pt", device=device) as f:
        for key in f.keys():
            tensor = f.get_tensor(key)
            state_dict[key] = tensor
    print(f"Successfully loaded {len(state_dict)} tensors.")
    return state_dict
chk = load_safetensors_to_torch("./temp_model/t3_cfg.safetensors")
import torch
embed = chk["text_emb.weight"]
embed2 = chk["text_head.weight"]
new_vocab_size = max(idx for idx in tokenizer_new.tokenizer.get_vocab().values()) + 1
new_embed = torch.zeros(new_vocab_size, 1024)
new_embed2 = torch.zeros(new_vocab_size, 1024)


new_embed[:704] = embed[:704].clone()
new_embed2[:704] = embed2[:704].clone()

tot = 0
# Iterate by actual token ID (0 to vocab_size-1)
for token_id in range(new_vocab_size):
    if token_id < 704:
        continue
        
    # Get token string using token ID (correct method)
    ru = tokenizer_new.decode([token_id])
    word = transliterate_ru_to_latin(ru)
    
    try:
        ids = tokenizer_orig.encode(word)
        if not ids:  # Handle empty encoding
            raise ValueError("Empty encoding result")
            
        # Initialize accumulators
        sume = torch.zeros(1024)
        sume2 = torch.zeros(1024)
        
        # Sum embeddings for all subtokens
        for i in ids:
            sume += embed[i]
            sume2 += embed2[i]
            
        # Store averaged embedding
        new_embed[token_id] = sume / len(ids)
        new_embed2[token_id] = sume2 / len(ids)
        
    except Exception as e:
        print(f"Failed with token {word}, {ru}: {str(e)}")
        tot += 1

# Update checkpoint with new embeddings
chk["text_emb.weight"] = new_embed
chk["text_head.weight"] = new_embed2
save_file(chk, "./temp_model/t3_cfg.safetensors")

In [ ]:
tokenizer_new.encode("привет")

In [ ]:
new_embed[802]

In [ ]:
tokenizer_orig.encode("privet")

In [ ]:
(embed[76] + embed[33]) /2

In [ ]:
%cp en_ru_tokenizer.json -d ./temp_model/tokenizer.json

In [ ]:
new_vocab_size

In [ ]:
%%writefile ./src/chatterbox/models/t3/modules/t3_config.py
from ..llama_configs import LLAMA_CONFIGS


class T3Config:
    def __init__(self, text_tokens_dict_size=911):
        self.start_text_token = 255
        self.stop_text_token = 0
        self.text_tokens_dict_size = text_tokens_dict_size
        self.max_text_tokens = 2048

        self.start_speech_token = 6561
        self.stop_speech_token = 6562
        self.speech_tokens_dict_size = 8194
        self.max_speech_tokens = 4096

        self.llama_config_name = "Llama_520M"
        self.input_pos_emb = "learned"
        self.speech_cond_prompt_len = 150

        self.encoder_type = "voice_encoder"
        self.speaker_embed_size = 256
        self.use_perceiver_resampler = True
        self.emotion_adv = True

    @property
    def n_channels(self):
        return LLAMA_CONFIGS[self.llama_config_name]["hidden_size"]
    
    @property
    def is_multilingual(self):
        return self.text_tokens_dict_size == 2454

    @classmethod
    def english_only(cls):
        """Create configuration for English-only TTS model."""
        return cls(text_tokens_dict_size=911)
    
    @classmethod 
    def multilingual(cls):
        """Create configuration for multilingual TTS model."""
        return cls(text_tokens_dict_size=2454)


## restart notebook

In [ ]:
%cd chatterbox

In [ ]:
import torchaudio as ta
from src.chatterbox.tts import ChatterboxTTS

model = ChatterboxTTS.from_local(ckpt_dir="./temp_model",device="cuda")


In [ ]:
text = "меня зовут Аммар"
wav = model.generate(text)
ta.save("test-ru.wav", wav, model.sr)